# Ground Magnetics: Reading, Correcting and Plotting Station Data

**Near-Surface Geophysics — Missouri S&T**

A magnetometer measures the strength of the Earth's magnetic field, in
nanotesla (nT). Near the surface that field is about 50 000 nT, and anything
magnetic in the ground — a buried steel drum, a pipe, a basalt dyke, fired
brick — adds a few to a few thousand nT on top of it. A magnetic survey maps
those small additions, the **anomalies**.

The field also changes on its own through the day — the **diurnal
variation**, tens of nT on a quiet day — and a magnetometer cannot tell that
change from geology. Removing it needs repeat readings at one fixed
**base station** through the survey.

This notebook:

1. **Reads** a table of station values.
2. **Corrects for the diurnal variation** from the base-station readings —
   or, when there are none, makes sure no unverifiable correction is used.
3. **Checks the data quality**: how well the repeat readings at each station
   agreed.
4. **Plots** it: a profile per line, all lines on one plan view, and a
   contour map of the anomaly.

With no file of your own it runs on a **synthetic survey**: two buried
objects under nine lines, with a base station re-read before every line.

**Units.** Field values are in nT. Distances are plotted in the unit your
file uses (feet here); `shallowgeo` stores them internally in metres.

## 0. Setup

Run this cell first. It installs `shallowgeo` if it is missing (or out of
date) and imports what the notebook uses.

In [ ]:
# Checks for the *magnetics* code specifically: an older install imports
# fine and then lacks it, and since the version string does not change
# between builds, a plain `pip install` reports "already satisfied".
import importlib
import sys

PACKAGE = ("shallow-geophysics @ "
           "git+https://github.com/Maurer-GEMLab/shallow-geophysics.git")

def shallowgeo_ready():
    for name in [n for n in sys.modules
                 if n == "shallowgeo" or n.startswith("shallowgeo.")]:
        del sys.modules[name]
    importlib.invalidate_caches()
    try:
        from shallowgeo.magnetics import diurnal_correction, synthetic_table  # noqa: F401
        from shallowgeo.drivers.mag_summary import read_mag_summary  # noqa: F401
        return True
    except ImportError:
        return False

if not shallowgeo_ready():
    %pip install -q --no-cache-dir "$PACKAGE"
if not shallowgeo_ready():
    %pip install -q --force-reinstall --no-deps --no-cache-dir "$PACKAGE"
if not shallowgeo_ready():
    raise ImportError(
        "shallowgeo.magnetics is still missing after installing. "
        "In Colab: Runtime > Restart session, then run this cell again."
    )

from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import shallowgeo as sg
from shallowgeo import magnetics as mag
from shallowgeo.drivers.mag_summary import inspect_mag_summary

plt.rcParams.update({"figure.dpi": 110})
pd.set_option("display.width", 120)
print("ready")

## 1. The data

Leave `DATA = None` to use the synthetic survey. To use your own table, set
`DATA` to its path. In Colab, upload the file with the file browser on the
left (or mount Drive) and give its path, e.g.
`DATA = Path("/content/my_mag_summary.csv")`.

The reader expects **one row per station reading**. It recognises these
columns by their header labels (capitalisation and unit suffixes such as
`_ft` or `_nT` do not matter):

| Column | Example label | Needed? |
|---|---|---|
| line number | `Line_No` | optional (one line if absent) |
| station number | `Station` | this or distance |
| distance along the line | `Dist_from_S_ft` | for every plot — the unit comes from the label |
| time of the reading | `Time` (`13:42:10`) | for a proper diurnal correction |
| number of readings averaged | `N_Readings` | optional |
| station mean, as recorded | `Mean_nT` | needed for a diurnal correction |
| standard deviation of the readings | `StdDev_nT` | optional, but it is the quality check |
| correction added to the line | `Correction_nT` | optional |
| final value | `Leveled_Mean_nT` | optional |

Every station appears once, **except the base station**, which appears once
for every time it was read.

Three settings go with the file:

- **`BASE_STATION`** — the station you returned to through the day, as
  `"L1-S1"` or `(1, 1)` (line, station). `None` if there was no base station.
- **`DIURNAL`** — what to do about the diurnal variation:
  - `"base"`: compute the correction from the base-station readings. The
    right choice whenever you have them.
  - `"remove"`: use the station means as recorded and ignore any correction
    column in the file. **The right choice when no base-station readings
    were taken** — a correction that did not come from base readings cannot
    be checked, and is better left out than trusted.
  - `"file"`: use the file's corrected values as they are.
- **`LINE_SPACING`** — distance between neighbouring lines, in the file's
  distance unit. It is not in the table, and the map cannot be drawn without
  it. Negative if the line numbers increase toward the west.
- **`DATE`** — the survey date, if the time column has none.

In [ ]:
DATA = None                              # or Path("/content/my_mag_summary.csv")
BASE_STATION = None                      # e.g. "L1-S1" or (1, 1); None if none was read
DIURNAL = "base" if BASE_STATION else "remove"
LINE_SPACING = 10.0                      # in the file's distance unit -- set yours
DATE = None                              # e.g. "2026-10-12"

if DATA is None:
    # The synthetic survey: base station at line 1 station 1, re-read
    # before every line and at the end.
    DATA = Path("synthetic_mag_survey.csv")
    mag.synthetic_table().to_csv(DATA, index=False)
    BASE_STATION, DIURNAL, LINE_SPACING, DATE = (1, 1), "base", 10.0, "2026-10-12"
    print(f"Using the synthetic survey, written to {DATA}")

# How each column header will be read. Check this before trusting anything below.
inspect_mag_summary(DATA)

If a column comes back with a `name` that is not in the table above, it is
carried along but not used. Rename the header in your file if it should
have been recognised.

In [ ]:
survey = sg.read(DATA, base_station=BASE_STATION, diurnal=DIURNAL,
                 line_spacing=LINE_SPACING, date=DATE)

r = survey.readings
print(f"{r['station_id'].nunique()} stations on {r['line'].nunique()} lines, "
      f"{len(r)} rows")
print(f"distances in {survey.metadata['distance_units_in_file']}, "
      f"measured from the {survey.metadata['distance_from'] or 'start'} end")
print(f"diurnal: {survey.metadata['diurnal']!r}"
      + (f", base station {survey.metadata['base_station']}" if survey.metadata["base_station"] else ""))
print(f"field values: {r['value'].min():,.0f} to {r['value'].max():,.0f} nT "
      f"(median {r['value'].median():,.0f})")

stations = mag.station_table(survey)
stations.head(10)

## 2. The diurnal correction

**With a base station** (`DIURNAL = "base"`), the base readings show how
much the field drifted between visits. The drift is interpolated to the
time of every other reading and subtracted, reducing the whole survey to
the field as it was at the first base reading. The plot below shows the
base readings and the drift curve drawn through them. It should be smooth:
a jump between two base readings is a change of set-up (sensor height, a
truck parked nearby), not the Sun.

If the table has no times, the drift is interpolated by reading order
instead, which assumes a steady walking pace — the reader warns when it
does this.

**Without a base station** (`DIURNAL = "remove"`), nothing is corrected and
the values are the station means as recorded. Any correction column in the
file is kept in the table for reference but not used.

In [ ]:
if survey.provenance.applied("diurnal"):
    mag.plot_diurnal(survey)
    plt.show()
    step = [s for s in survey.provenance if s.operation == "diurnal"][-1].parameters
    print(f"{step['n_base']} base readings; largest drift {step['max_drift_nT']:.1f} nT; "
          f"{step['readings_outside_base_span']} readings outside the base readings' time span "
          "(those got the nearest base value)")
elif survey.provenance.applied("file_correction"):
    print("Using the correction column from the file, unchecked.")
    if "correction" in survey.readings:
        mag.plot_corrections(survey)
else:
    print("No diurnal correction: values are the station means as recorded.")
    if "correction" in survey.readings:
        print("The file's correction column is kept, but not used:")
        mag.plot_corrections(survey)

**Is the correction big enough to matter?** Compare the largest drift above
with the anomalies in the plots below. A drift of 30 nT matters for a
survey looking for a 20 nT archaeological feature, and is invisible beside
a 2000 nT buried drum.

## 3. How well did the repeat readings agree?

Two readings a few seconds apart at the same spot should agree to within a
few nT. When they do not, something close to the sensor is distorting the
field — a steel fence post, the operator's keys, a buried pipe directly
below, or the sensor being moved between readings in a strong gradient. The
station value is then an average of two different things, and should be
trusted least.

`MAX_SIGMA` sets the threshold for calling a station **noisy**. 100 nT is a
starting point; look at the plot and decide whether there is a natural
break.

In [ ]:
MAX_SIGMA = 100.0   # nT

if "repeat_sigma" in survey.readings:
    mag.plot_repeatability(survey, max_sigma=MAX_SIGMA)
    plt.show()
    noisy = stations[stations["noisy"]].sort_values("repeat_sigma", ascending=False)
    print(f"{len(noisy)} of {len(stations)} stations have repeat sigma > {MAX_SIGMA:g} nT")
    display(noisy.head(15))
else:
    print("No standard-deviation column in the file, so no repeat-reading check.")

## 4. Profiles: one panel per line

The first plot to look at. Each panel is one line, field against distance.
The blue curve is the corrected value with ± one repeat standard deviation
as faint bars. The dashed grey curve is the value before the diurnal
correction, so the gap between the two is the correction. Noisy stations
are circled in red.

All panels share one field scale so anomaly sizes can be compared between
lines. Set `sharey=False` to let each line fill its panel, or
`lines=[3, 4, 5]` to look at a few.

In [ ]:
mag.plot_profiles(survey, show_raw=True, max_sigma=MAX_SIGMA);

**Things to look for.**

- A **smooth rise or fall** over many stations is a deep or broad source (or
  the regional field). A **spike at one station** is a shallow, compact
  source — or a bad reading. The repeat sigma helps tell them apart: a real
  shallow source gives a large value that both readings agree on.
- A spike circled in red is suspect. One that is *not* circled, and appears
  at the same distance on the next line too, is probably real.
- Over a magnetic object the total field usually shows a **pair**: a high on
  one side and a low on the other. At mid-northern latitudes (Missouri is
  about 37° N, where the field dips about 64° below horizontal) the high
  lies to the south of the source and the low to the north.

## 5. All lines on one plan view

Each line is drawn at its across-line position, and the field along it as a
wiggle: swinging right (shaded red) where the field is above the survey's
median and left (shaded blue) where it is below. An anomaly from one buried
object should appear on several neighbouring lines at the same distance.
This plot needs no interpolation, so nothing on it is invented.

In [ ]:
mag.plot_stacked_profiles(survey, max_sigma=MAX_SIGMA);

## 6. Contour map

To draw a map, values have to be **interpolated** between stations. Here the
stations are joined into triangles and the field varies linearly across each
triangle, so the map stops at the edge of the survey rather than inventing
values beyond it.

The map shows the **anomaly**: the field minus the survey's median, on a
colour scale centred on zero (red above, blue below). Subtracting a constant
is enough to remove the Earth's main field over a site this small; its
change across a few hundred feet is under a nanotesla.

Two versions side by side, on the same colour scale: every station, and with
the noisy stations left out of the gridding. Features that survive in both
are the ones to believe. Interpolation is only as good as the line spacing:
an anomaly smaller than the gap between lines can fall between them
entirely, and one that crosses a single line gets smeared across to its
neighbours.

In [ ]:
fig, (a1, a2) = plt.subplots(1, 2, figsize=(13, 7))
mag.plot_map(survey, max_sigma=MAX_SIGMA, ax=a1)
a1.set_title("All stations", loc="left", fontsize=10)
mag.plot_map(survey, max_sigma=MAX_SIGMA, drop_noisy=True, ax=a2)
a2.set_title(f"Stations with repeat sigma > {MAX_SIGMA:g} nT left out", loc="left", fontsize=10)
fig.tight_layout()

The same data as **total field** rather than anomaly, on a one-colour
scale:

In [ ]:
mag.plot_map(survey, anomaly=False, max_sigma=MAX_SIGMA, drop_noisy=True);

## 7. Save the results

The station table (with the noisy flag and the diurnal correction applied
at each station) and the map, to hand in or to open in a spreadsheet. In
Colab, the files appear in the file browser on the left; right-click to
download.

In [ ]:
OUT = Path("magnetics_output")
OUT.mkdir(exist_ok=True)

mag.station_table(survey, max_sigma=MAX_SIGMA).to_csv(OUT / "stations.csv", index=False)
ax = mag.plot_map(survey, max_sigma=MAX_SIGMA, drop_noisy=True)
ax.figure.savefig(OUT / "anomaly_map.png", dpi=200, bbox_inches="tight")
plt.close(ax.figure)
print("wrote", sorted(p.name for p in OUT.iterdir()))

## 8. Questions

1. Re-read the data with `diurnal="remove"` and redraw the profiles. How
   large is the diurnal correction compared with the anomalies? Would the
   map look different without it?
2. Which line has the most noisy stations? Look at where they are along the
   line. Is there anything at the site that could explain them?
3. Pick the largest anomaly on the map. Does it appear on more than one line
   in the plan-view plot (section 5)? Is there a high–low pair, and is the
   high on the south side?
4. Change `LINE_SPACING` to twice its value and redraw the map. Which
   features change shape, and why? What does that tell you about how far
   apart lines should be to map an object of a given size?
5. How often would you go back to the base station on your next survey?
   Look at how much the drift changed between two base readings in
   section 2.